# Optimal Execution Strategies

This notebook demonstrates optimal execution theory and implementation:
- **Almgren-Chriss Model**: Optimal execution balancing impact and risk
- **Price Impact Models**: Permanent and temporary impact
- **Execution Strategies**: Comparison of aggressive vs passive approaches
- **Cost Analysis**: Trading costs and slippage
- **Trajectory Optimization**: Finding the optimal trade schedule

These techniques minimize execution costs for large orders.

In [ ]:
# Import required libraries
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize

# Import custom modules
from utils import generate_price_series
from models import (
    AlmgrenChrissModel,
    LinearPriceImpactModel,
    compare_execution_strategies,
    simulate_execution_with_impact
)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print('✓ Libraries loaded successfully')

## 1. Problem Setup

Define the execution problem: large order to be executed over time.

In [ ]:
# Execution parameters
total_quantity = 100000  # Total shares to execute
total_time = 10.0  # Execution horizon (e.g., 10 time periods)
num_periods = 20  # Number of execution slices
initial_price = 100.0  # Starting price

# Market parameters
volatility = 0.02  # Price volatility per period
permanent_impact = 0.1  # Permanent price impact coefficient
temporary_impact = 0.05  # Temporary price impact coefficient
risk_aversion = 1e-6  # Risk aversion parameter (lambda)

print('Execution Problem Setup:')
print('=' * 70)
print(f'Total quantity: {total_quantity:,} shares')
print(f'Execution horizon: {total_time} periods')
print(f'Number of slices: {num_periods}')
print(f'Initial price: ${initial_price:.2f}')
print(f'\nMarket parameters:')
print(f'  Volatility: {volatility:.4f}')
print(f'  Permanent impact: {permanent_impact:.4f}')
print(f'  Temporary impact: {temporary_impact:.4f}')
print(f'  Risk aversion: {risk_aversion:.2e}')
print('=' * 70)

## 2. Almgren-Chriss Optimal Execution

Implement the Almgren-Chriss model for optimal trade scheduling.

In [ ]:
# Create Almgren-Chriss model
ac_model = AlmgrenChrissModel(
    total_quantity=total_quantity,
    total_time=total_time,
    volatility=volatility,
    permanent_impact=permanent_impact,
    temporary_impact=temporary_impact,
    risk_aversion=risk_aversion
)

# Calculate optimal trajectory
optimal_trajectory = ac_model.calculate_optimal_trajectory(num_periods)

print('Almgren-Chriss Optimal Trajectory:')
print('=' * 70)
print(optimal_trajectory.head(10))
print('\n...')
print(optimal_trajectory.tail(5))
print('=' * 70)

# Calculate expected costs
costs = ac_model.calculate_expected_cost(optimal_trajectory)

print('\nExpected Execution Costs:')
print('=' * 70)
for cost_type, value in costs.items():
    print(f'{cost_type:.<30s} {value:>15,.2f}')
print('=' * 70)

## 3. Visualize Optimal Trajectory

Show how the optimal strategy distributes trades over time.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Holdings over time
axes[0].plot(optimal_trajectory['time'], optimal_trajectory['holdings'], 
             linewidth=2, marker='o', markersize=6, label='Optimal Holdings')
axes[0].fill_between(optimal_trajectory['time'], 0, optimal_trajectory['holdings'], alpha=0.3)
axes[0].set_ylabel('Shares Remaining')
axes[0].set_title('Optimal Holdings Trajectory (Almgren-Chriss)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:,.0f}'))

# Trade sizes
axes[1].bar(optimal_trajectory['time'][:-1], optimal_trajectory['trade_size'][:-1], 
            width=total_time/(num_periods*1.2), alpha=0.7, edgecolor='black')
avg_trade = optimal_trajectory['trade_size'][:-1].mean()
axes[1].axhline(avg_trade, color='r', linestyle='--', 
                linewidth=2, label=f'Average: {avg_trade:,.0f}')
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Trade Size (shares)')
axes[1].set_title('Optimal Trade Schedule')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:,.0f}'))

plt.tight_layout()
plt.show()

print('\n✓ Optimal trajectory visualization complete')
print(f'\nKey observation: Optimal strategy trades more aggressively early')
print(f'This balances permanent impact (favors slower) vs timing risk (favors faster)')

## 4. Compare Execution Strategies

Compare optimal execution with alternative strategies.

In [ ]:
# Define alternative strategies
strategies = {}

# 1. Uniform (TWAP-like)
uniform_trades = np.ones(num_periods) * total_quantity / num_periods
strategies['Uniform'] = pd.DataFrame({
    'time': optimal_trajectory['time'][:-1],
    'trade_size': uniform_trades
})

# 2. Front-loaded (aggressive)
weights = np.array([2 * i / (num_periods * (num_periods + 1)) for i in range(num_periods, 0, -1)])
front_loaded = weights * total_quantity
strategies['Front-loaded'] = pd.DataFrame({
    'time': optimal_trajectory['time'][:-1],
    'trade_size': front_loaded
})

# 3. Back-loaded (passive)
weights = np.array([2 * i / (num_periods * (num_periods + 1)) for i in range(1, num_periods + 1)])
back_loaded = weights * total_quantity
strategies['Back-loaded'] = pd.DataFrame({
    'time': optimal_trajectory['time'][:-1],
    'trade_size': back_loaded
})

# 4. Optimal (Almgren-Chriss)
strategies['Optimal (AC)'] = pd.DataFrame({
    'time': optimal_trajectory['time'][:-1],
    'trade_size': optimal_trajectory['trade_size'][:-1].values
})

print('Strategy Comparison:')
print('=' * 70)
for name, strategy in strategies.items():
    print(f'\n{name}:')
    print(f'  Total quantity: {strategy["trade_size"].sum():,.0f} shares')
    print(f'  Average trade: {strategy["trade_size"].mean():,.0f} shares')
    print(f'  Std dev: {strategy["trade_size"].std():,.0f} shares')
    print(f'  First trade: {strategy["trade_size"].iloc[0]:,.0f} shares')
    print(f'  Last trade: {strategy["trade_size"].iloc[-1]:,.0f} shares')

In [ ]:
# Visualize strategy comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, (name, strategy) in enumerate(strategies.items()):
    axes[idx].bar(strategy['time'], strategy['trade_size'], 
                  width=total_time/(num_periods*1.2), alpha=0.7, edgecolor='black')
    avg = strategy['trade_size'].mean()
    axes[idx].axhline(avg, color='r', linestyle='--', linewidth=2, label=f'Avg: {avg:,.0f}')
    axes[idx].set_xlabel('Time')
    axes[idx].set_ylabel('Trade Size (shares)')
    axes[idx].set_title(f'{name} Strategy')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)
    axes[idx].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:,.0f}'))

plt.tight_layout()
plt.show()

## 5. Price Impact Model

Analyze how trade size affects prices through market impact.

In [ ]:
# Create price impact model
impact_model = LinearPriceImpactModel(
    permanent_coef=permanent_impact,
    temporary_coef=temporary_impact
)

# Calculate impact for different trade sizes
trade_sizes = np.linspace(0, 10000, 50)
impacts = []

for size in trade_sizes:
    impact = impact_model.calculate_impact(size, direction=1)
    impacts.append({
        'trade_size': size,
        'permanent_impact': impact['permanent_impact'],
        'temporary_impact': impact['temporary_impact'],
        'total_impact': impact['total_impact']
    })

impact_df = pd.DataFrame(impacts)

print('Price Impact Analysis:')
print('=' * 70)
print(f'Model: Linear impact (Impact = coefficient × trade_size)')
print(f'Permanent coefficient: {permanent_impact}')
print(f'Temporary coefficient: {temporary_impact}')
print(f'\nExample: Trade of 5,000 shares')
sample_impact = impact_model.calculate_impact(5000, direction=1)
print(f'  Permanent impact: ${sample_impact["permanent_impact"]:.2f}')
print(f'  Temporary impact: ${sample_impact["temporary_impact"]:.2f}')
print(f'  Total impact: ${sample_impact["total_impact"]:.2f}')
print(f'  Impact as % of $100 price: {sample_impact["total_impact"]:.3%}')

In [ ]:
# Visualize price impact
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Impact components
axes[0].plot(impact_df['trade_size'], impact_df['permanent_impact'], 
             linewidth=2, label='Permanent Impact', marker='o', markersize=4)
axes[0].plot(impact_df['trade_size'], impact_df['temporary_impact'], 
             linewidth=2, label='Temporary Impact', marker='s', markersize=4)
axes[0].plot(impact_df['trade_size'], impact_df['total_impact'], 
             linewidth=3, label='Total Impact', color='black', linestyle='--')
axes[0].set_xlabel('Trade Size (shares)')
axes[0].set_ylabel('Price Impact ($)')
axes[0].set_title('Price Impact Components')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Impact as percentage of price
impact_pct = (impact_df['total_impact'] / initial_price) * 100
axes[1].plot(impact_df['trade_size'], impact_pct, linewidth=2, color='red')
axes[1].fill_between(impact_df['trade_size'], 0, impact_pct, alpha=0.3, color='red')
axes[1].set_xlabel('Trade Size (shares)')
axes[1].set_ylabel('Impact (% of price)')
axes[1].set_title('Total Price Impact as Percentage')
axes[1].grid(True, alpha=0.3)
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.show()

## 6. Simulate Execution with Price Impact

Run simulations to see realized costs under different strategies.

In [ ]:
# Simulate execution for each strategy
np.random.seed(42)
simulation_results = {}

for name, strategy in strategies.items():
    # Convert strategy to proper format for simulation
    execution_schedule = pd.DataFrame({
        'period': range(len(strategy)),
        'trade_size': strategy['trade_size'].values
    })
    
    # Run simulation
    results = simulate_execution_with_impact(
        execution_schedule=execution_schedule,
        initial_price=initial_price,
        volatility=volatility,
        impact_model=impact_model,
        seed=42
    )
    
    simulation_results[name] = results
    
    # Calculate metrics
    total_cost = results['realized_cost'].sum()
    avg_price = (results['price'] * results['trade_size']).sum() / results['trade_size'].sum()
    cost_bps = (avg_price - initial_price) / initial_price * 10000
    
    print(f'\n{name}:')
    print(f'  Average execution price: ${avg_price:.2f}')
    print(f'  Total realized cost: ${total_cost:,.2f}')
    print(f'  Cost vs initial (bps): {cost_bps:.2f}')

print('\n' + '=' * 70)

In [ ]:
# Visualize execution simulations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, (name, results) in enumerate(simulation_results.items()):
    axes[idx].plot(results['price'], linewidth=2, marker='o', markersize=6, label='Execution Price')
    axes[idx].axhline(initial_price, color='green', linestyle='--', linewidth=2, label='Initial Price')
    final_price = results['price'].iloc[-1]
    axes[idx].axhline(final_price, color='red', linestyle='--', linewidth=2, 
                      label=f'Final: ${final_price:.2f}')
    axes[idx].set_xlabel('Trade Number')
    axes[idx].set_ylabel('Price ($)')
    axes[idx].set_title(f'{name} - Price Evolution')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Cost Analysis and Comparison

Comprehensive comparison of execution costs across strategies.

In [ ]:
# Calculate comprehensive cost metrics
cost_comparison = []

for name, results in simulation_results.items():
    total_cost = results['realized_cost'].sum()
    total_value = (results['price'] * results['trade_size']).sum()
    avg_price = total_value / results['trade_size'].sum()
    
    cost_comparison.append({
        'Strategy': name,
        'Avg Price': avg_price,
        'Total Cost': total_cost,
        'Cost (bps)': (avg_price - initial_price) / initial_price * 10000,
        'Total Value': total_value,
        'Price Impact': results['price'].iloc[-1] - initial_price
    })

cost_df = pd.DataFrame(cost_comparison)
cost_df = cost_df.sort_values('Total Cost')

print('\nExecution Cost Comparison:')
print('=' * 80)
print(cost_df.to_string(index=False))
print('=' * 80)

best_strategy = cost_df.iloc[0]['Strategy']
worst_strategy = cost_df.iloc[-1]['Strategy']
cost_savings = cost_df.iloc[-1]['Total Cost'] - cost_df.iloc[0]['Total Cost']

print(f'\nBest strategy: {best_strategy}')
print(f'Worst strategy: {worst_strategy}')
print(f'Potential savings: ${cost_savings:,.2f} ({cost_savings/cost_df.iloc[-1]["Total Cost"]*100:.1f}%)')

In [ ]:
# Visualize cost comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Total costs
colors = ['green' if i == 0 else 'red' if i == len(cost_df)-1 else 'blue' 
          for i in range(len(cost_df))]
axes[0].bar(cost_df['Strategy'], cost_df['Total Cost'], color=colors, alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Total Cost ($)')
axes[0].set_title('Total Execution Costs')
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'${y:,.0f}'))
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha='right')

# Cost in basis points
axes[1].bar(cost_df['Strategy'], cost_df['Cost (bps)'], color=colors, alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Cost (basis points)')
axes[1].set_title('Execution Cost vs Initial Price')
axes[1].grid(True, alpha=0.3, axis='y')
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha='right')

# Average execution price
axes[2].bar(cost_df['Strategy'], cost_df['Avg Price'], color=colors, alpha=0.7, edgecolor='black')
axes[2].axhline(initial_price, color='black', linestyle='--', linewidth=2, label='Initial Price')
axes[2].set_ylabel('Average Price ($)')
axes[2].set_title('Average Execution Price')
axes[2].legend()
axes[2].grid(True, alpha=0.3, axis='y')
plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 8. Sensitivity Analysis

Examine how costs change with different market parameters.

In [ ]:
# Test different risk aversion levels
risk_aversions = [1e-7, 1e-6, 1e-5, 1e-4]
sensitivity_results = []

for lambda_risk in risk_aversions:
    model = AlmgrenChrissModel(
        total_quantity=total_quantity,
        total_time=total_time,
        volatility=volatility,
        permanent_impact=permanent_impact,
        temporary_impact=temporary_impact,
        risk_aversion=lambda_risk
    )
    
    trajectory = model.calculate_optimal_trajectory(num_periods)
    costs = model.calculate_expected_cost(trajectory)
    
    first_trade_pct = trajectory['trade_size'].iloc[0] / total_quantity * 100
    
    sensitivity_results.append({
        'Risk Aversion': lambda_risk,
        'Total Cost': costs['total_expected_cost'],
        'Permanent Cost': costs['permanent_cost'],
        'Temporary Cost': costs['temporary_cost'],
        'Timing Risk': costs['timing_risk'],
        'First Trade %': first_trade_pct
    })

sensitivity_df = pd.DataFrame(sensitivity_results)

print('Sensitivity to Risk Aversion:')
print('=' * 80)
print(sensitivity_df.to_string(index=False))
print('=' * 80)
print('\nObservation: Higher risk aversion → trade faster → higher impact cost')

In [ ]:
# Visualize sensitivity
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Cost components vs risk aversion
x_labels = [f'{x:.0e}' for x in sensitivity_df['Risk Aversion']]
x_pos = np.arange(len(x_labels))
width = 0.25

axes[0].bar(x_pos - width, sensitivity_df['Permanent Cost'], width, 
            label='Permanent', alpha=0.8)
axes[0].bar(x_pos, sensitivity_df['Temporary Cost'], width, 
            label='Temporary', alpha=0.8)
axes[0].bar(x_pos + width, sensitivity_df['Timing Risk'], width, 
            label='Timing Risk', alpha=0.8)
axes[0].set_xlabel('Risk Aversion Parameter')
axes[0].set_ylabel('Cost')
axes[0].set_title('Cost Components vs Risk Aversion')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(x_labels)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# First trade percentage
axes[1].plot(x_labels, sensitivity_df['First Trade %'], 
             marker='o', markersize=10, linewidth=2, color='purple')
axes[1].set_xlabel('Risk Aversion Parameter')
axes[1].set_ylabel('First Trade (% of total)')
axes[1].set_title('Aggressiveness vs Risk Aversion')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Conclusion

In this notebook, we've explored optimal execution strategies:

### Key Concepts:

**1. Almgren-Chriss Model**
- Balances market impact vs timing risk
- Produces optimal trade schedule
- Risk aversion parameter controls urgency
- Generally front-loads trades to reduce timing risk

**2. Price Impact**
- Permanent impact: Long-lasting price change
- Temporary impact: Short-term liquidity cost
- Linear model: Impact proportional to trade size
- Larger trades have higher per-share costs

**3. Strategy Comparison**
- Uniform (TWAP): Simple but suboptimal
- Front-loaded: Fast but high impact
- Back-loaded: Patient but high risk
- Optimal: Balances tradeoffs

**4. Cost Components**
- Market impact (permanent + temporary)
- Timing risk (price volatility)
- Opportunity cost
- Total cost depends on strategy and parameters

### Key Insights:
- Optimal execution significantly reduces costs
- Strategy choice depends on:
  - Order size relative to market
  - Market volatility
  - Risk tolerance
  - Time constraints
- Mathematical optimization provides quantitative framework
- Real-world implementation requires:
  - Parameter estimation from data
  - Adaptation to changing conditions
  - Integration with risk limits

### Practical Applications:
- Portfolio rebalancing
- Liquidation of large positions
- Institutional order execution
- Algorithmic trading strategies

### Extensions:
- Non-linear impact models
- Time-varying volatility
- Multiple assets
- Order book dynamics
- Machine learning for parameter estimation